# 03 — Federated K-Means Runs

Run federated k-means via FeatureCloud and aggregate results.

**What this notebook does:**
1. Prepare per-site input directories for the FeatureCloud fc_kmeans app.
2. Launch federated k-means tests (requires Docker + FeatureCloud CLI).
3. OR aggregate existing federated outputs (if FeatureCloud is unavailable).
4. Join federated cluster assignments with metadata.

**Prerequisites:**
- Run notebook 01 first (data preparation).
- For live runs: Docker running + FeatureCloud CLI installed.
- For aggregation only: existing zip results in `real_datasets/<dataset>/inputs/*/tests/`.

## Imports

In [1]:
import sys
from pathlib import Path
import pandas as pd

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from evaluation_utils.real_datasets_utils import (
    dataset_configs,
    discover_clients,
    load_feature_matrix,
    load_metadata,
    prepare_variant_inputs,
    aggregate_fed_clusters,
)
from evaluation_utils.featurecloud_kmeans_utils import (
    ensure_app_image,
    run_single_federated_variant,
    aggregate_existing_federated_output,
)

## Configuration

In [2]:
DATASETS = ["ecoli", "ovarian_cancer", "quartet", "ccRCC_proteomics", "multiomics"]

# Set to True to launch a real FeatureCloud test.
# Set to False to only aggregate existing zip results.
RERUN_FEDERATED = True

# FeatureCloud settings (only needed when RERUN_FEDERATED=True)
APP_IMAGE = "fc_kmeans_upd"
APP_SOURCE_DIR = NOTEBOOK_DIR.parent / "federated_kmeans_upd"
CONTROLLER_HOST = "http://localhost:8000"
QUERY_INTERVAL = 5
TIMEOUT = 1800

OUTPUT_ROOT = NOTEBOOK_DIR

## Prepare FeatureCloud Inputs

Build per-site input directories with `intensities.tsv`, `design.tsv`,
and `config_kmeans.yml` — the format required by the fc_kmeans app.

This step creates `real_datasets/<dataset>/inputs/{before,corrected}/<site>/`
from the prepared matrices saved by notebook 01. It always runs (fast and idempotent)
so that the federated step below has up-to-date inputs.

In [3]:
# Always generate per-site input directories (fast, idempotent).
# These are needed for both fresh federated runs and for aggregating existing results.
configs = dataset_configs(REPO_ROOT)

for ds_name in DATASETS:
    cfg = configs[ds_name]
    ds_dir = OUTPUT_ROOT / ds_name
    prepared_dir = ds_dir / "prepared"

    # Load prepared data from notebook 01
    before = load_feature_matrix(prepared_dir / "before_matrix.tsv")
    corrected = load_feature_matrix(prepared_dir / "corrected_matrix.tsv")
    metadata = pd.read_csv(prepared_dir / "metadata.tsv", sep="\t")
    clients = discover_clients(cfg)

    k_condition = int(metadata['condition'].nunique())
    k_batch = int(metadata['lab'].nunique())
    k_values = sorted(set([k_condition, k_batch]))

    print(f"\n{'='*60}")
    print(f"{ds_name}: preparing FC inputs for k={k_values}")
    print(f"{'='*60}")

    # Generate per-site input directories for the fc_kmeans app
    before_dir = prepare_variant_inputs(
        dataset_root=ds_dir, variant_name="before",
        matrix=before, metadata=metadata,
        clients=clients, k_values=k_values,
    )
    corrected_dir = prepare_variant_inputs(
        dataset_root=ds_dir, variant_name="corrected",
        matrix=corrected, metadata=metadata,
        clients=clients, k_values=k_values,
    )
    print(f"Created: {before_dir}")
    print(f"Created: {corrected_dir}")


ecoli: preparing FC inputs for k=[2, 5]
Created: /home/yuliya-cosybio/repos/cosybio/fedRBE/evaluation_clusterization_after_correction/real_datasets/ecoli/inputs/before
Created: /home/yuliya-cosybio/repos/cosybio/fedRBE/evaluation_clusterization_after_correction/real_datasets/ecoli/inputs/corrected

ovarian_cancer: preparing FC inputs for k=[2, 6]
Created: /home/yuliya-cosybio/repos/cosybio/fedRBE/evaluation_clusterization_after_correction/real_datasets/ovarian_cancer/inputs/before
Created: /home/yuliya-cosybio/repos/cosybio/fedRBE/evaluation_clusterization_after_correction/real_datasets/ovarian_cancer/inputs/corrected

quartet: preparing FC inputs for k=[4]
Created: /home/yuliya-cosybio/repos/cosybio/fedRBE/evaluation_clusterization_after_correction/real_datasets/quartet/inputs/before
Created: /home/yuliya-cosybio/repos/cosybio/fedRBE/evaluation_clusterization_after_correction/real_datasets/quartet/inputs/corrected

ccRCC_proteomics: preparing FC inputs for k=[2, 3]
Created: /home/yul

## Run or Aggregate Federated K-Means

When `RERUN_FEDERATED=True`, this launches a FeatureCloud test for each variant.
When `False`, it aggregates existing zip results from previous runs.

In [4]:
for ds_name in DATASETS:
    cfg = configs[ds_name]
    ds_dir = OUTPUT_ROOT / ds_name
    metadata = pd.read_csv(ds_dir / "prepared" / "metadata.tsv", sep="\t")
    clients = discover_clients(cfg)
    client_names = [c.name for c in clients]

    k_condition = int(metadata['condition'].nunique())
    k_batch = int(metadata['lab'].nunique())
    k_values = sorted(set([k_condition, k_batch]))

    print(f"\n{'='*60}")
    print(f"{ds_name}: {'running' if RERUN_FEDERATED else 'aggregating'} federated k-means")
    print(f"{'='*60}")

    for variant in ["before", "corrected"]:
        variant_input_dir = ds_dir / "inputs" / variant

        try:
            output_path = run_single_federated_variant(
                dataset_name=ds_name,
                variant_label=variant,
                variant_input_dir=variant_input_dir,
                dataset_root=ds_dir,
                metadata=metadata,
                client_names=client_names,
                k_values=k_values,
                app_image=APP_IMAGE,
                controller_host=CONTROLLER_HOST,
                query_interval=QUERY_INTERVAL,
                timeout=TIMEOUT,
                keep_extracted=False,
                aggregate_only=not RERUN_FEDERATED,
            )
            print(f"  {variant}: saved to {output_path}")
        except FileNotFoundError as e:
            print(f"  {variant}: SKIPPED — {e}")
        except RuntimeError as e:
            print(f"  {variant}: FAILED — {e}")


ecoli: running federated k-means
[ecoli] Starting FeatureCloud controller for 'before' data
Killing leftover containers...
Starting the FeatureCloud controller with the data directory /home/yuliya-cosybio/repos/cosybio/fedRBE/evaluation_clusterization_after_correction/real_datasets/ecoli/inputs/before


Downloading...: 148it [00:06, 23.22it/s]


Controller started successfully, waiting 5s to ensure the controller is properly running!
[ecoli] Running federated k-means for 'before' data
_______________EXPERIMENT_______________
Test started successfully!
Test finished successfully!
instances: [{'id': 0, 'name': 'fc_fckmeansupd_547192681', 'containerID': '8c02d5622c512318', 'coordinator': False, 'frontendUrl': '/app-traffic/f7b1fc5a30/', 'message': 'terminal', 'state': None, 'progress': 1}, {'id': 1, 'name': 'fc_fckmeansupd_76486068', 'containerID': '791e4c8b8ac870e1', 'coordinator': True, 'frontendUrl': '/app-traffic/d39c7a3e8f/', 'message': 'terminal', 'state': None, 'progress': 1}, {'id': 2, 'name': 'fc_fckmeansupd_992823757', 'containerID': '7155b1e182fdb02b', 'coordinator': False, 'frontendUrl': '/app-traffic/1785afd6f1/', 'message': 'terminal', 'state': None, 'progress': 1}, {'id': 3, 'name': 'fc_fckmeansupd_865297513', 'containerID': '0e1d93ccef4df6a0', 'coordinator': False, 'frontendUrl': '/app-traffic/09a9669687/', 'messa

Downloading...: 3it [00:00, 2359.00it/s]


Controller started successfully, waiting 5s to ensure the controller is properly running!
[ecoli] Running federated k-means for 'corrected' data
_______________EXPERIMENT_______________
Test started successfully!
Test finished successfully!
instances: [{'id': 0, 'name': 'fc_fckmeansupd_286716565', 'containerID': '1749c27c0660b9b8', 'coordinator': False, 'frontendUrl': '/app-traffic/09385e3ce6/', 'message': 'terminal', 'state': None, 'progress': 1}, {'id': 1, 'name': 'fc_fckmeansupd_262748873', 'containerID': 'fbed72869064b3b2', 'coordinator': False, 'frontendUrl': '/app-traffic/56d22b48fe/', 'message': 'terminal', 'state': None, 'progress': 1}, {'id': 2, 'name': 'fc_fckmeansupd_313823817', 'containerID': '3fc155de6bbb6fb8', 'coordinator': True, 'frontendUrl': '/app-traffic/33b0f3d409/', 'message': 'terminal', 'state': None, 'progress': 1}, {'id': 3, 'name': 'fc_fckmeansupd_385785466', 'containerID': 'e762ff99c6e8dac5', 'coordinator': False, 'frontendUrl': '/app-traffic/2faf5b96b7/', 'm

...

Controller started successfully, waiting 5s to ensure the controller is properly running!
[quartet] Running federated k-means for 'corrected' data
_______________EXPERIMENT_______________
Test started successfully!
Test finished successfully!
instances: [{'id': 0, 'name': 'fc_fckmeansupd_695298249', 'containerID': 'b5355cdf464ec049', 'coordinator': False, 'frontendUrl': '/app-traffic/09c4e2bcce/', 'message': 'terminal', 'state': None, 'progress': 1}, {'id': 1, 'name': 'fc_fckmeansupd_637216722', 'containerID': '9c60f1113da92dca', 'coordinator': False, 'frontendUrl': '/app-traffic/caf5e6975e/', 'message': 'terminal', 'state': None, 'progress': 1}, {'id': 2, 'name': 'fc_fckmeansupd_669842749', 'containerID': '6093a2b05f059178', 'coordinator': True, 'frontendUrl': '/app-traffic/86b8d92096/', 'message': 'terminal', 'state': None, 'progress': 1}, {'id': 3, 'name': 'fc_fckmeansupd_672175812', 'containerID': 'eaf645aa3121700d', 'coordinator': False, 'frontendUrl': '/app-traffic/12275e6b4d/', 

Downloading...: 3it [00:00, 1716.87it/s]


Controller started successfully, waiting 5s to ensure the controller is properly running!
[ccRCC_proteomics] Running federated k-means for 'before' data
_______________EXPERIMENT_______________
Test started successfully!
Test finished successfully!
instances: [{'id': 0, 'name': 'fc_fckmeansupd_543678902', 'containerID': '880be2b52cfb5fa6', 'coordinator': True, 'frontendUrl': '/app-traffic/d030bff511/', 'message': 'terminal', 'state': None, 'progress': 1}, {'id': 1, 'name': 'fc_fckmeansupd_586384831', 'containerID': '52ebacc75b2d1d5a', 'coordinator': False, 'frontendUrl': '/app-traffic/38297b1328/', 'message': 'terminal', 'state': None, 'progress': 1}, {'id': 2, 'name': 'fc_fckmeansupd_554596629', 'containerID': '150381a8a4e99f47', 'coordinator': False, 'frontendUrl': '/app-traffic/3b3486ed47/', 'message': 'terminal', 'state': None, 'progress': 1}]
TEST DONE:
_______________EXPERIMENT FINISHED SUCCESSFULLY_______________
[ccRCC_proteomics] Federated output saved: /home/yuliya-cosybio/re

Downloading...: 3it [00:00, 1434.28it/s]


Controller started successfully, waiting 5s to ensure the controller is properly running!
[ccRCC_proteomics] Running federated k-means for 'corrected' data
_______________EXPERIMENT_______________
Test started successfully!
Test finished successfully!
instances: [{'id': 0, 'name': 'fc_fckmeansupd_336162230', 'containerID': 'c5706f046a3fd910', 'coordinator': True, 'frontendUrl': '/app-traffic/75be85d7aa/', 'message': 'terminal', 'state': None, 'progress': 1}, {'id': 1, 'name': 'fc_fckmeansupd_332312447', 'containerID': '61f7e1e8f84bfe41', 'coordinator': False, 'frontendUrl': '/app-traffic/1b9745a942/', 'message': 'terminal', 'state': None, 'progress': 1}, {'id': 2, 'name': 'fc_fckmeansupd_297459235', 'containerID': '08280f39585541c0', 'coordinator': False, 'frontendUrl': '/app-traffic/9b712c0627/', 'message': 'terminal', 'state': None, 'progress': 1}]
TEST DONE:
_______________EXPERIMENT FINISHED SUCCESSFULLY_______________
[ccRCC_proteomics] Federated output saved: /home/yuliya-cosybio

## Verify Outputs

Check that federated metadata files were produced.

In [5]:
for ds_name in DATASETS:
    run_dir = OUTPUT_ROOT / ds_name / "kmeans_res" / "runs"
    for fname in ["1_metadata_before_fedclusters.tsv", "1_metadata_after_fedclusters.tsv"]:
        p = run_dir / fname
        if p.exists():
            df = pd.read_csv(p, sep="\t")
            print(f"{ds_name}/{fname}: {df.shape[0]} samples, cols={list(df.columns)}")
        else:
            print(f"{ds_name}/{fname}: NOT FOUND")

ecoli/1_metadata_before_fedclusters.tsv: 118 samples, cols=['file', 'condition', 'lab', 'Fed_2clusters', 'Fed_5clusters']
ecoli/1_metadata_after_fedclusters.tsv: 118 samples, cols=['file', 'condition', 'lab', 'Fed_2clusters', 'Fed_5clusters']
ovarian_cancer/1_metadata_before_fedclusters.tsv: 332 samples, cols=['file', 'condition', 'lab', 'Fed_2clusters', 'Fed_6clusters']
ovarian_cancer/1_metadata_after_fedclusters.tsv: 332 samples, cols=['file', 'condition', 'lab', 'Fed_2clusters', 'Fed_6clusters']
quartet/1_metadata_before_fedclusters.tsv: 72 samples, cols=['file', 'condition', 'lab', 'Fed_4clusters']
quartet/1_metadata_after_fedclusters.tsv: 72 samples, cols=['file', 'condition', 'lab', 'Fed_4clusters']
ccRCC_proteomics/1_metadata_before_fedclusters.tsv: 887 samples, cols=['file', 'condition', 'lab', 'Fed_2clusters', 'Fed_3clusters']
ccRCC_proteomics/1_metadata_after_fedclusters.tsv: 887 samples, cols=['file', 'condition', 'lab', 'Fed_2clusters', 'Fed_3clusters']
